# 一步步构建循环神经网络

欢迎来到课程 5 的第一个作业，你将在 NumPy 中实现循环神经网络或 RNN 的关键组件！ 

读完本作业，你将能够：

* 定义构建序列模型的符号
* 描述基本 RNN 的架构
* 识别 LSTM 的主要组件
* 随着时间的推移，为基本的 RNN 和 LSTM 实现反向传播
* 举例RNN的几种类型 

循环神经网络 (RNN) 对于自然语言处理和其他序列任务非常有效，因为它们具有“记忆”。它们可以一次读取一个输入 $x^{\langle t \rangle}$（例如单词），并通过隐藏层激活值记住一些从一个时间步传递到下一个时间步的上下文信息。这允许单向（单向）RNN 从过去获取信息来处理以后的输入。双向（双向）RNN 可以从过去和未来获取上下文，就像 Marty McFly 一样。 

**符号**：
- 上标$[l]$表示与$l^{th}$层关联的对象。 

- 上标 $(i)$ 表示与 $i^{th}$ 示例关联的对象。 

- 上标$\langle t \rangle$表示$t^{th}$时间的对象 
步骤。 
    
- 下标$i$表示向量的$i^{th}$条目。

**示例**：  
- $a^{(2)[3]<4>}_5$ 表示向量中第 2 个训练样本 (2)、第 3 层 [3]、第 4 个时间步 <4> 和第 5 个条目的激活值。

#### 先决条件
* 你应该已经熟悉`numpy`
* 如果需要复习 NumPy，可以回顾本专项课程的第 1 门课“神经网络与深度学习”：
    * 具体回顾一下第2周的练习作业 ["Python Basics with Numpy (optional assignment)"](https://www.coursera.org/learn/neural-networks-deep-learning/programming/isoAV/python-basics-with-numpy)
    
    
#### 修改起始代码时要小心！
* 在处理计分函数时，请记住仅修改以下代码之间的代码：
```Python
#### START CODE HERE
```
 和：
```Python
#### END CODE HERE
```
* 特别要避免修改计分函数的第一行。这些始于：
```Python
# GRADED FUNCTION: routine_name
```
自动评分器（AutoGrader）依靠这些标记定位函数，因此即使只改变空格也可能导致评分失败。如果标记被修改或缺失，评分器会返回 failed。现在开始吧！

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/nlp-sequence-models/supplement/qHIve/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [依赖库](#0)
- [1 - 基本循环神经网络的前向传播](#1)
    - [1.1 - RNN 单元](#1-1)
        - [练习 1 - rnn_cell_forward](#ex-1)
    - [1.2 - RNN 前向传播](#1-2)
        - [练习 2 - rnn_forward](#ex-2)
- [2 - 长短期记忆 (LSTM) 网络](#2)
    - [2.1 - LSTM 单元](#2-1)
        - [练习 3 - lstm_cell_forward](#ex-3)
    - [2.2 - LSTM 的前向传播](#2-2)
        - [练习 4 - lstm_forward](#ex-4)
- [3 - 反向传播 in 循环神经网络 (OPTIONAL / UNGRADED)](#3)
    - [3.1 - 基本 RNN 反向传播](#3-1)
        - [练习 5 - rnn_cell_backward](#ex-5)
        - [练习 6 - rnn_backward](#ex-6)
    - [3.2 - LSTM 反向传播](#3-2)
        - [练习 7 - lstm_cell_backward](#ex-7)
    - [3.3 LSTM 循环神经网络的反向传播](#3-3)
        - [练习 8 - lstm_backward](#ex-8)

<a name='0'></a>
## 依赖库

In [ ]:
### v2.2

In [ ]:
import numpy as np
from rnn_utils import *
from public_tests import *

<a name='1'></a>
## 1 - 基本循环神经网络的前向传播

本周晚些时候，你将有机会使用 RNN 生成音乐！你将实现的基本 RNN 具有以下结构： 

在此示例中，$T_x = T_y$。 

<img src="images/RNN.png" style="width:500px;height:300px;">
<div style="text-align:center;color:purple;"><b>图1</b>：基本RNN模型</div>

### 输入尺寸 $x$

#### 包含 $n_x$ 个单元的输入
* 对于单个输入示例的单个时间步，$x^{(i) \langle t \rangle }$ 是一维输入向量
* 以语言为例，具有 5000 个单词词汇的语言可以被 one-hot 编码为具有 5000 个单元的向量。  所以 $x^{(i)\langle t \rangle}$ 的形状为 (5000,)  
* 此处使用符号 $n_x$ 来表示单个训练样本的单个时间步中的单位数

#### 长度为 $T_{x}$ 的时间序列
* 循环神经网络有多个时间步，你将使用 $t$ 对其进行索引。
* 在课程中，你看到了一个由多个时间步 $T_x$ 组成的训练样本 $x^{(i)}$。在此 Notebook 中，$T_{x}$ 将表示最长序列中的时间步数。

#### 批量大小为 $m$
* 假设我们有 mini-batch，每个有 20 个训练样本  
* 要从向量化中受益，你将堆叠 20 列 $x^{(i)}$ 示例
* 例如，这个张量的形状为 (5000,20,10) 
* 你将使用 $m$ 来表示训练样本的编号  
* 所以，mini-batch 的形状是 $(n_x,m,T_x)$

#### 形状为 $(n_{x},m,T_{x})$ 的 3D 张量
* 形状为 $(n_x,m,T_x)$ 的 3 维张量 $x$ 表示输入到 RNN 的输入 $x$

#### 为每个时间步截取一个 2D 切片：$x^{\langle t \rangle}$
* 在每个时间步，你将使用训练样本的 mini-batch （不仅仅是一个示例）
* 因此，对于每个时间步 $t$，你将使用形状为 $(n_x,m)$ 的 2D 切片
* 该 2D 切片称为 $x^{\langle t \rangle}$。  代码中的变量名称为`xt`。

### 隐藏状态 $a$ 的定义

* 从一个时间步传递到另一个时间步到 RNN 的激活值 $a^{\langle t \rangle}$ 称为“隐藏状态”。

### 隐藏状态 $a$ 的尺寸

* 与输入张量 $x$ 类似，单个训练样本的隐藏状态是长度为 $n_{a}$ 的向量
* 如果包含 $m$ 训练样本的 mini-batch，则 mini-batch 的形状为 $(n_{a},m)$
* 当包含时间步维度时，隐藏状态的形状为 $(n_{a}, m, T_x)$
* 你将循环遍历索引为 $t$ 的时间步，并使用 3D 张量的 2D 切片  
* 该 2D 切片称为 $a^{\langle t \rangle}$
* 在代码中，使用的变量名称是 `a_prev` 或 `a_next`，具体取决于正在实现的函数
* 这个2D切片的形状是$(n_{a}, m)$

### 预测维度 $\hat{y}$
* 与输入和隐藏状态类似，$\hat{y}$ 是形状为 $(n_{y}, m, T_{y})$ 的 3D 张量
    * $n_{y}$：表示预测的向量中的单元数
    * $m$：mini-batch 中的示例数
    * $T_{y}$：预测中的时间步数
* 对于单个时间步 $t$，2D 切片 $\hat{y}^{\langle t \rangle}$ 的形状为 $(n_{y}, m)$
* 代码中，变量名称为：
    - `y_pred`: $\hat{y}$ 
    - `yt_pred`: $\hat{y}^{\langle t \rangle}$

以下是实现 RNN 的方法： 

### 步骤：
1. 实现 RNN 一个时间步所需的计算。
2. 在 $T_x$ 时间步上实现一个循环，以便一次处理一个输入。 

<a name='1-1'></a>
### 1.1 - RNN 单元

你可以将循环神经网络视为单个单元的重复使用。首先，你将实现单个时间步的计算。下图描述了 RNN 单元的单个时间步的操作： 

<img src="images/rnn_step_forward_figure2_v3a.png" style="width:700px;height:300px;">
<div style="text-align:center;color:purple;">

<b> 图 2</b>：基本 RNN 单元。以 $x^{\langle t \rangle}$（当前输入）和 $a^{\langle t - 1\rangle}$（包含过去信息的前一个隐藏状态）作为输入，并输出 $a^{\langle t \rangle}$，该输出被赋予下一个 RNN 单元格，也用于预测 $\hat{y}^{\langle t \rangle}$

</div>

**`RNN cell` 与 `RNN_cell_forward`**：
* 请注意，RNN 单元输出隐藏状态 $a^{\langle t \rangle}$。  
    * `RNN cell` 图中为实线内盒  
* 你将实现的函数 `rnn_cell_forward` 也会计算预测 $\hat{y}^{\langle t \rangle}$
    * `RNN_cell_forward` 图中虚线为外盒

<a name='ex-1'></a>
### 练习 1 - rnn_cell_forward

实现图 2 中描述的 RNN 单元。

**说明**：
1. 使用 tanh 激活值计算隐藏状态：$a^{\langle t \rangle} = \tanh(W_{aa} a^{\langle t-1 \rangle} + W_{ax} x^{\langle t \rangle} + b_a)$
2. 使用新的隐藏状态 $a^{\langle t \rangle}$，计算预测 $\hat{y}^{\langle t \rangle} = softmax(W_{ya} a^{\langle t \rangle} + b_y)$。 （提供函数 `softmax`）
3. 将 $(a^{\langle t \rangle}, a^{\langle t-1 \rangle}, x^{\langle t \rangle}, parameters)$ 存储在 `cache` 中
4. 返回 $a^{\langle t \rangle}$ 、 $\hat{y}^{\langle t \rangle}$ 和 `cache`

#### 附加提示
* 关于 [numpy.tanh](https://numpy.org/devdocs/reference/generated/numpy.tanh.html) 的更多信息
* 在这个作业中，有一个现有的`softmax` 函数供你使用。  它位于文件“rnn_utils.py”中并且已被导入。
* 对于矩阵乘法，使用 [numpy.dot](https://docs.scipy.org/doc/numpy/reference/generated/numpy.dot.html)


In [ ]:
# UNQ_C1 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: rnn_cell_forward

def rnn_cell_forward(xt, a_prev, parameters):
    """
    实现 RNN 单元的单个前向步骤，如图 (2) 所示

    参数：
    xt -- 时间步 "t" 的输入数据，为形状 (n_x, m) 的 NumPy 数组。
    a_prev -- 隐藏状态，时间步 "t-1"，NumPy 形状数组 (n_a, m)
    parameters -- Python字典包含：
                        Wax -- 权重矩阵乘以输入，NumPy 形状为 (n_a, n_x) 的数组
                        Waa -- 权重矩阵乘以形状为 (n_a, n_a) 的隐藏状态、NumPy 数组
                        Wya -- 将隐藏状态与输出相关的权重矩阵，形状为 (n_y, n_a) 的 NumPy 数组
                        ba --  偏置，NumPy 形状数组 (n_a, 1)
                        by -- 输出层的偏置，将隐藏状态映射至输出时使用，为形状 (n_y, 1) 的 NumPy 数组
    返回：
    a_next -- 下一个隐藏状态，形状为 (n_a, m)
    yt_pred -- 时间步预测 "t", NumPy 形状数组 (n_y, m)
    cache -- 反向传播所需的值的元组，包含 (a_next, a_prev, xt, parameters)
    """
    
    # 从"parameters"中检索参数
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]
    
    ### START CODE HERE ### (≈2 lines)
    # 使用上面给出的公式计算下一个激活值状态
    a_next = None
    # 使用上面给出的公式计算当前单元格的输出
    yt_pred = None
    ### END CODE HERE ###
    
    # 将反向传播所需的值存储在缓存中
    cache = (a_next, a_prev, xt, parameters)
    
    return a_next, yt_pred, cache

In [ ]:
np.random.seed(1)
xt_tmp = np.random.randn(3, 10)
a_prev_tmp = np.random.randn(5, 10)
parameters_tmp = {}
parameters_tmp['Waa'] = np.random.randn(5, 5)
parameters_tmp['Wax'] = np.random.randn(5, 3)
parameters_tmp['Wya'] = np.random.randn(2, 5)
parameters_tmp['ba'] = np.random.randn(5, 1)
parameters_tmp['by'] = np.random.randn(2, 1)

a_next_tmp, yt_pred_tmp, cache_tmp = rnn_cell_forward(xt_tmp, a_prev_tmp, parameters_tmp)
print("a_next[4] = \n", a_next_tmp[4])
print("a_next.shape = \n", a_next_tmp.shape)
print("yt_pred[1] =\n", yt_pred_tmp[1])
print("yt_pred.shape = \n", yt_pred_tmp.shape)

# 单元测试
rnn_cell_forward_tests(rnn_cell_forward)

**预期输出**： 
```Python
a_next[4] = 
 [ 0.59584544  0.18141802  0.61311866  0.99808218  0.85016201  0.99980978
 -0.18887155  0.99815551  0.6531151   0.82872037]
a_next.shape = 
 (5, 10)
yt_pred[1] =
 [ 0.9888161   0.01682021  0.21140899  0.36817467  0.98988387  0.88945212
  0.36920224  0.9966312   0.9982559   0.17746526]
yt_pred.shape = 
 (2, 10)

```

<a name='1-2'></a>
### 1.2 - RNN 前向传播 

- 循环神经网络 (RNN) 是你刚刚构建的 RNN 单元的重复。 
    - 如果你的输入数据序列有 10 个时间步，那么你将重复使用 RNN 单元 10 次 
- 每个单元格在每个时间步接受两个输入：
    - $a^{\langle t-1 \rangle}$：上一个单元格中的隐藏状态
    - $x^{\langle t \rangle}$：当前时间步的输入数据
- 每个时间步有两个输出：
    - A 隐藏状态 ($a^{\langle t \rangle}$)
    - 预测 ($y^{\langle t \rangle}$)
- 每个时间步重用权重和偏置 $(W_{aa}, W_{ax}, b_{a}, W_{ay}, b_{y})$ 
    - 它们在“参数”字典中调用 `rnn_cell_forward` 之间进行维护


<img src="images/rnn_forward_sequence_figure3_v3a.png" style="width:800px;height:180px;">
<div style="text-align:center;color:purple;">

<b>图3</b>：基本RNN。输入序列 $x = (x^{\langle 1 \rangle}, x^{\langle 2 \rangle}, ..., x^{\langle T_x \rangle})$ 通过 $T_x$ 时间步进行。网络输出 $y = (y^{\langle 1 \rangle}, y^{\langle 2 \rangle}, ..., y^{\langle T_x \rangle})$。

</div>


<a name='ex-2'></a>
### 练习 2 - rnn_forward

实现图 3 中描述的 RNN 的前向传播。

**说明**：
* 创建一个 3D 零数组，形状为 $(n_{a}, m, T_{x})$ 的 $a$，它将存储由 RNN 计算的所有隐藏状态
* 创建一个 3D 零数组 $\hat{y}$，形状为 $(n_{y}, m, T_{x})$，用于存储预测  
    - 请注意，在本例中，$T_{y} = T_{x}$（预测和输入具有相同的时间步数）
* 初始化 2D 隐藏状态 `a_next`，将其设置为等于初始隐藏状态、$a_{0}$
* 在每个时间步 $t$：
    - 获取 $x^{\langle t \rangle}$，这是 $x$ 单个时间步 $t$ 的 2D 切片
        - $x^{\langle t \rangle}$ 的形状为 $(n_{x}, m)$
        - $x$ 的形状为 $(n_{x}, m, T_{x})$
    - 通过运行 `rnn_cell_forward` 更新 2D 隐藏状态 $a^{\langle t \rangle}$（变量名称 `a_next`）、预测 $\hat{y}^{\langle t \rangle}$ 和缓存
        - $a^{\langle t \rangle}$ 的形状为 $(n_{a}, m)$
    - 将 2D 隐藏状态存储在 3D 张量 $a$ 中的 $t^{th}$ 位置
        - $a$ 的形状为 $(n_{a}, m, T_{x})$
    - 将 2D $\hat{y}^{\langle t \rangle}$ 预测（变量名称 `yt_pred`）存储在 3D 张量 $\hat{y}_{pred}$ 的 $t^{th}$ 位置
        - $\hat{y}^{\langle t \rangle}$ 的形状为 $(n_{y}, m)$
        - $\hat{y}$ 的形状为 $(n_{y}, m, T_x)$
    - 将缓存追加到缓存列表中
* 返回3D张量$a$和$\hat{y}$，以及缓存列表

#### 附加提示
- [np.zeros](https://docs.scipy.org/doc/numpy/reference/generated/numpy.zeros.html) 的一些有用文档
- 如果你有一个 3 维 NumPy 数组并按其第三维进行索引，则可以像这样使用数组切片：`var_name[:,:,i]`

In [ ]:
# UNQ_C2 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: rnn_forward

def rnn_forward(x, a0, parameters):
    """
    实现图（3）中描述的循环神经网络的前向传播。

    参数：
    x -- 每个时间步的输入数据，形状为 (n_x, m, T_x)。
    a0 -- 初始隐藏状态，形状为 (n_a, m)
    parameters -- Python字典包含：
                        Waa -- 权重矩阵乘以形状为 (n_a, n_a) 的隐藏状态、NumPy 数组
                        Wax -- 权重矩阵乘以输入，NumPy 形状为 (n_a, n_x) 的数组
                        Wya -- 将隐藏状态与输出相关的权重矩阵，形状为 (n_y, n_a) 的 NumPy 数组
                        ba --  偏置 NumPy 形状数组 (n_a, 1)
                        by -- 输出层的偏置，将隐藏状态映射至输出时使用，为形状 (n_y, 1) 的 NumPy 数组

    返回：
    a -- 每个时间步的隐藏状态，形状为 (n_a, m, T_x) 的 NumPy 数组
    y_pred -- 每个时间步的预测，NumPy 形状数组 (n_y, m, T_x)
    caches -- 反向传播所需的值的元组，包含（缓存列表，x）
    """
    
    # 初始化"caches"，其中将包含所有缓存的列表
    caches = []
    
    # 从 x 的形状和参数中检索尺寸["Wya"]
    n_x, m, T_x = x.shape
    n_y, n_a = parameters["Wya"].shape
    
    ### START CODE HERE ###
    
    # 用零初始化 "a" 和 "y_pred"（约 2 行）
    a = None
    y_pred = None
    
    # 初始化a_next（约1行）
    a_next = None
    
    # 循环所有时间步
    for t in range(T_x):
        # 更新下一个隐藏状态，计算预测，获取缓存（约1行）
        a_next, yt_pred, cache = None
        # 将新的"next" 隐藏状态的值保存在一个（≈1行）中
        a[:,:,t] = None
        # 将预测值保存在 y（≈1 行）中
        y_pred[:,:,t] = None
        # 将 "cache" 附加到 "caches"（约 1 行）
        caches.append(None)
    ### END CODE HERE ###
    
    # 将反向传播所需的值存储在缓存中
    caches = (caches, x)
    
    return a, y_pred, caches

In [ ]:
np.random.seed(1)
x_tmp = np.random.randn(3, 10, 4)
a0_tmp = np.random.randn(5, 10)
parameters_tmp = {}
parameters_tmp['Waa'] = np.random.randn(5, 5)
parameters_tmp['Wax'] = np.random.randn(5, 3)
parameters_tmp['Wya'] = np.random.randn(2, 5)
parameters_tmp['ba'] = np.random.randn(5, 1)
parameters_tmp['by'] = np.random.randn(2, 1)

a_tmp, y_pred_tmp, caches_tmp = rnn_forward(x_tmp, a0_tmp, parameters_tmp)
print("a[4][1] = \n", a_tmp[4][1])
print("a.shape = \n", a_tmp.shape)
print("y_pred[1][3] =\n", y_pred_tmp[1][3])
print("y_pred.shape = \n", y_pred_tmp.shape)
print("caches[1][1][3] =\n", caches_tmp[1][1][3])
print("len(caches) = \n", len(caches_tmp))

#单元测试    
rnn_forward_test(rnn_forward)

**预期输出**：

```Python
a[4][1] = 
 [-0.99999375  0.77911235 -0.99861469 -0.99833267]
a.shape = 
 (5, 10, 4)
y_pred[1][3] =
 [ 0.79560373  0.86224861  0.11118257  0.81515947]
y_pred.shape = 
 (2, 10, 4)
caches[1][1][3] =
 [-1.1425182  -0.34934272 -0.20889423  0.58662319]
len(caches) = 
 2
```

### 恭喜！ 

你已经从头开始成功构建了循环神经网络的前向传播。干得好！ 

#### RNN 表现更好的情况：
- 这对于某些应用程序来说已经足够好了，但它会受到梯度消失的影响。 
- 当每个输出 $\hat{y}^{\langle t \rangle}$ 可以使用“本地”上下文进行估计时，RNN 效果最佳。  
- “局部”上下文是指靠近预测时间步 $t$ 的信息。
- 更正式地说，本地上下文是指输入 $x^{\langle t' \rangle}$ 和预测 $\hat{y}^{\langle t \rangle}$，其中 $t'$ 接近 $t$。

<div style="color:blue;">

<b>你应该记住：</b>
* 循环神经网络，或RNN，本质上是单个单元的重复使用。
* 基本的 RNN 一次读取一个输入，并通过从一个时间步传递到下一个时间步的隐藏层激活值 (隐藏状态) 记住信息。
    * 时间步维度决定了RNN单元的重复使用次数
* 每个单元格在每个时间步接受两个输入：
    * 上一个单元格中的隐藏状态
    * 当前时间步的输入数据
* 每个单元在每个时间步都有两个输出：
    * A 隐藏状态 
    * 预测


</div>

在下一节中，你将构建一个更复杂的模型 LSTM，它更擅长寻址梯度消失。 LSTM 能够更好地记住一条信息并将其保存多个时间步。  

<a name='2'></a>
## 2 - 长短期记忆 (LSTM) 网络

下图展示了LSTM小区的操作：

<img src="images/LSTM_figure4_v3a.png" style="width:500px;height:400px;">
<div style="text-align:center;color:purple;">

<b>图4</b>：LSTM单元。这会在每个时间步跟踪并更新“单元状态”或内存变量 $c^{\langle t \rangle}$，这可能与 $a^{\langle t \rangle}$ 不同。  
注意，$softmax^{}$ 包括全连接层和 softmax.

</div>

与上面的 RNN 示例类似，你将首先为单个时间步实现 LSTM 单元。然后，你将从“for 循环”内部迭代调用它，以使其以 $T_x$ 时间步处理输入。 

### 门和状态概述

#### 遗忘门$\mathbf{\Gamma}_{f}$

* 假设你正在阅读一段文本中的单词，并计划使用 LSTM 来跟踪语法结构，例如主语是单数（“puppy”）还是复数（“puppies”）。 
* 如果主体改变了它的状态（从单数词到复数词），之前状态的记忆就会过时，所以你会“忘记”那个过时的状态。
* “忘记门”是一个包含 0 到 1 之间值的张量。
    * 如果忘记门中的单元的值接近0，则LSTM将“忘记”前一个单元状态的相应单元中存储的状态。
    * 如果遗忘门中的某个单元的值接近 1，则 LSTM 将主要记住存储状态中的相应值。

##### 方程

$$\mathbf{\Gamma}_f^{\langle t \rangle} = \sigma(\mathbf{W}_f[\mathbf{a}^{\langle t-1 \rangle}, \mathbf{x}^{\langle t \rangle}] + \mathbf{b}_f)\tag{1} $$

##### 等式解释：

* $\mathbf{W_{f}}$ 包含控制遗忘门行为的权重。 
* 前一个时间步的隐藏状态 $[a^{\langle t-1 \rangle}$ 和当前时间步的输入 $x^{\langle t \rangle}]$ 连接在一起并乘以 $\mathbf{W_{f}}$。 
* sigmoid 函数用于使每个门张量的值 $\mathbf{\Gamma}_f^{\langle t \rangle}$ 的范围从 0 到 1。
* 遗忘门 $\mathbf{\Gamma}_f^{\langle t \rangle}$ 与之前的单元状态 $c^{\langle t-1 \rangle}$ 具有相同的维度。 
* 这意味着两者可以按元素相乘。
* 将张量 $\mathbf{\Gamma}_f^{\langle t \rangle} * \mathbf{c}^{\langle t-1 \rangle}$ 相乘就像在之前的单元状态上应用掩码一样。
* 如果 $\mathbf{\Gamma}_f^{\langle t \rangle}$ 中的单个值是 0 或接近 0，则乘积接近 0。
    * 这可以防止存储在 $\mathbf{c}^{\langle t-1 \rangle}$ 中相应单元中的信息在下一个时间步中被记住。
* 同样，如果一个值接近 1，则乘积接近前一单元状态中的原始值。
    * LSTM 将保留 $\mathbf{c}^{\langle t-1 \rangle}$ 对应单元的信息，以供下一个时间步使用。
    
##### 代码中的变量名
代码中的变量名称与方程类似，略有差异。  
* `Wf`：忘记门权重 $\mathbf{W}_{f}$
* `bf`：忘记栅极偏置 $\mathbf{b}_{f}$
* `ft`：忘记门$\Gamma_f^{\langle t \rangle}$

#### 候选值 $\tilde{\mathbf{c}}^{\langle t \rangle}$
* 候选值是一个张量，包含当前时间步的信息，**可能**存储在当前单元状态 $\mathbf{c}^{\langle t \rangle}$ 中。
* 候选值中被传递的部分取决于更新门。
* 候选值是包含范围从 -1 到 1 的值的张量。
* 波形符“~”用于区分候选 $\tilde{\mathbf{c}}^{\langle t \rangle}$ 和单元状态 $\mathbf{c}^{\langle t \rangle}$。

##### 方程
$$\mathbf{\tilde{c}}^{\langle t \rangle} = \tanh\left( \mathbf{W}_{c} [\mathbf{a}^{\langle t - 1 \rangle}, \mathbf{x}^{\langle t \rangle}] + \mathbf{b}_{c} \right) \tag{3}$$

##### 等式的解释
* *tanh* 函数生成 -1 到 1 之间的值。


##### 代码中的变量名
* `cct`：候选值$\mathbf{\tilde{c}}^{\langle t \rangle}$

#### 更新门 $\mathbf{\Gamma}_{i}$

* 你使用更新门来决定将候选 $\tilde{\mathbf{c}}^{\langle t \rangle}$ 的哪些方面添加到单元状态 $c^{\langle t \rangle}$ 中。
* 更新门决定“候选”张量 $\tilde{\mathbf{c}}^{\langle t \rangle}$ 的哪些部分被传递到单元状态 $\mathbf{c}^{\langle t \rangle}$ 上。
* 更新门是一个包含 0 到 1 之间值的张量。
    * 当更新门中的一个单元接近1时，它允许将候选$\tilde{\mathbf{c}}^{\langle t \rangle}$的值传递到隐藏状态 $\mathbf{c}^{\langle t \rangle}$上
    * 当更新门中的某个单元接近0时，它会阻止候选中的相应值传递到隐藏状态上。
* 请注意，使用下标“i”而不是“u”，以遵循文献中使用的约定。

##### 方程

$$
\mathbf{\Gamma}_i^{\langle t \rangle} = \sigma(\mathbf{W}_i[a^{\langle t-1 \rangle}, \mathbf{x}^{\langle t \rangle}] + \mathbf{b}_i)\tag{2}
$$

##### 等式的解释

* 与遗忘门类似，这里的 $\mathbf{\Gamma}_i^{\langle t \rangle}$，sigmoid 生成 0 到 1 之间的值。
* 更新门与候选元素逐个相乘，该乘积（$\mathbf{\Gamma}_{i}^{\langle t \rangle} * \tilde{c}^{\langle t \rangle}$）用于确定单元状态 $\mathbf{c}^{\langle t \rangle}$。

##### 代码中的变量名称（请注意，它们与方程不同）
在代码中，你将使用学术文献中找到的变量名称。  这些变量不使用“u”来表示“更新”。
* `Wi` 是更新门权重 $\mathbf{W}_i$ （不是“Wu”） 
* `bi` 是更新栅极偏置 $\mathbf{b}_i$ （不是“bu”）
* `it` 是更新门 $\mathbf{\Gamma}_i^{\langle t \rangle}$ （不是“ut”）

#### 单元状态 $\mathbf{c}^{\langle t \rangle}$

* 单元状态是传递到未来时间步的“记忆”。
* 新的单元状态 $\mathbf{c}^{\langle t \rangle}$ 是先前单元状态和候选值的组合。

##### 方程

$$ \mathbf{c}^{\langle t \rangle} = \mathbf{\Gamma}_f^{\langle t \rangle}* \mathbf{c}^{\langle t-1 \rangle} + \mathbf{\Gamma}_{i}^{\langle t \rangle} *\mathbf{\tilde{c}}^{\langle t \rangle} \tag{4} $$

##### 等式解释
* 先前的单元状态 $\mathbf{c}^{\langle t-1 \rangle}$ 由遗忘门 $\mathbf{\Gamma}_{f}^{\langle t \rangle}$ 调整（加权）
* 和候选值 $\tilde{\mathbf{c}}^{\langle t \rangle}$，由更新门 $\mathbf{\Gamma}_{i}^{\langle t \rangle}$ 调整（加权）

##### 代码中的变量名称和形状
* `c`：单元状态，包括所有时间步，$\mathbf{c}$ 形状 $(n_{a}, m, T_x)$
* `c_next`：新的（下一个）单元状态，$\mathbf{c}^{\langle t \rangle}$ 形状 $(n_{a}, m)$
* `c_prev`：先前的单元状态，$\mathbf{c}^{\langle t-1 \rangle}$，形状$(n_{a}, m)$

#### 输出门 $\mathbf{\Gamma}_{o}$

* 输出门决定发送什么作为时间步的预测（输出）。
* 输出门与其他门类似，它包含范围从 0 到 1 的值。

##### 方程

$$
\mathbf{\Gamma}_o^{\langle t \rangle}=  \sigma(\mathbf{W}_o[\mathbf{a}^{\langle t-1 \rangle}, \mathbf{x}^{\langle t \rangle}] + \mathbf{b}_{o})\tag{5}
$$

##### 等式的解释
* 输出门由之前的隐藏状态 $\mathbf{a}^{\langle t-1 \rangle}$ 和当前的输入$\mathbf{x}^{\langle t \rangle}$ 决定
* sigmoid 使门范围从 0 到 1。


##### 代码中的变量名
* `Wo`：输出门权重，$\mathbf{W_o}$
* `bo`：输出栅极偏置，$\mathbf{b_o}$
* `ot`：输出门，$\mathbf{\Gamma}_{o}^{\langle t \rangle}$

#### 隐藏状态 $\mathbf{a}^{\langle t \rangle}$

* 隐藏状态被传递到 LSTM 单元的下一个时间步。
* 用于确定下一个时间步的三个门（$\mathbf{\Gamma}_{f}, \mathbf{\Gamma}_{u}, \mathbf{\Gamma}_{o}$）。
* 隐藏状态也用于预测 $y^{\langle t \rangle}$。

##### 方程

$$ \mathbf{a}^{\langle t \rangle} = \mathbf{\Gamma}_o^{\langle t \rangle} * \tanh(\mathbf{c}^{\langle t \rangle})\tag{6} $$

##### 等式解释
* 隐藏状态 $\mathbf{a}^{\langle t \rangle}$ 由单元状态 $\mathbf{c}^{\langle t \rangle}$ 结合输出门 $\mathbf{\Gamma}_{o}$ 确定。
* 单元状态通过 `tanh` 函数传递，以在 -1 和 1 之间重新调整值。
* 输出门就像一个“掩码”，它要么保留 $\tanh(\mathbf{c}^{\langle t \rangle})$ 的值，要么阻止这些值包含在隐藏状态 $\mathbf{a}^{\langle t \rangle}$ 中

##### 代码中的变量名称和形状
* `a`：隐藏状态，包括时间步。  $\mathbf{a}$ 具有形状 $(n_{a}, m, T_{x})$
* `a_prev`：来自上一个时间步的隐藏状态。 $\mathbf{a}^{\langle t-1 \rangle}$ 具有形状 $(n_{a}, m)$
* `a_next`：隐藏状态用于下一个时间步。  $\mathbf{a}^{\langle t \rangle}$ 具有形状 $(n_{a}, m)$ 

#### 预测$\mathbf{y}^{\langle t \rangle}_{pred}$
* 此用例中的预测是分类，因此你将使用 softmax。

等式为：
$$\mathbf{y}^{\langle t \rangle}_{pred} = \textrm{softmax}(\mathbf{W}_{y} \mathbf{a}^{\langle t \rangle} + \mathbf{b}_{y})$$

##### 代码中的变量名称和形状
* `y_pred`：预测，包括所有时间步。 $\mathbf{y}_{pred}$ 的形状为 $(n_{y}, m, T_{x})$。  请注意，本例中为 $(T_{y} = T_{x})$。
* `yt_pred`：当前时间步$t$的预测。 $\mathbf{y}^{\langle t \rangle}_{pred}$ 具有形状 $(n_{y}, m)$

<a name='2-1'></a>
### 2.1 - LSTM 单元

<a name='ex-3'></a>
### 练习 3 - lstm_cell_forward

实现图 4 中描述的 LSTM 单元。

**说明**：
1. 连接隐藏状态 $a^{\langle t-1 \rangle}$ 并将 $x^{\langle t \rangle}$ 输入到单个矩阵中：

$$
concat = \begin{bmatrix} a^{\langle t-1 \rangle} \\ x^{\langle t \rangle} \end{bmatrix}
$$

2. 计算门、隐藏状态和单元状态的所有公式（1 到 6）。
3. 计算预测 $y^{\langle t \rangle}$。


#### 附加提示
* 你可以使用[numpy.concatenate](https://docs.scipy.org/doc/numpy/reference/generated/numpy.concatenate.html)。  检查 `axis` 参数使用哪个值。
* 函数、`sigmoid()` 和 `softmax` 是从 `rnn_utils.py` 进口的。
* [numpy.tanh](https://docs.scipy.org/doc/numpy/reference/generated/numpy.tanh.html) 的一些文档
* 使用 [numpy.dot](https://docs.scipy.org/doc/numpy/reference/generated/numpy.dot.html) 进行矩阵乘法。
* 请注意，变量名称 `Wi`、`bi` 指的是 **更新** 门的权重和偏置。  函数中没有名为“Wu”或“bu”的变量。

In [ ]:
# UNQ_C3 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: lstm_cell_forward

def lstm_cell_forward(xt, a_prev, c_prev, parameters):
    """
    实现 LSTM 单元的单个前向步骤，如图 (4) 所示

    参数：
    xt -- 时间步 "t" 的输入数据，为形状 (n_x, m) 的 NumPy 数组。
    a_prev -- 隐藏状态，时间步 "t-1"，NumPy 形状数组 (n_a, m)
    c_prev -- 时间步 "t-1"、NumPy 形状数组的内存状态 (n_a, m)
    parameters -- Python字典包含：
                        Wf -- 遗忘门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        bf -- 遗忘门的偏置，NumPy 数组，形状为 (n_a, 1)
                        Wi -- 更新门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        bi -- 更新门的偏置，形状为 (n_a, 1) 的 NumPy 数组
                        Wc -- 第一个 "tanh"、NumPy 形状数组的权重矩阵 (n_a, n_a + n_x)
                        bc --  第一个 "tanh"、NumPy 形状数组的偏置 (n_a, 1)
                        Wo -- 输出门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        bo --  输出门的偏置，NumPy 数组，形状为 (n_a, 1)
                        Wy -- 将隐藏状态与输出相关的权重矩阵，形状为 (n_y, n_a) 的 NumPy 数组
                        by -- 输出层的偏置，将隐藏状态映射至输出时使用，为形状 (n_y, 1) 的 NumPy 数组
                        
    返回：
    a_next -- 下一个隐藏状态，形状为 (n_a, m)
    c_next -- 下一个内存状态，形状为 (n_a, m)
    yt_pred -- 时间步预测 "t", NumPy 形状数组 (n_y, m)
    cache -- 反向传播所需的值的元组，包含 (a_next, c_next, a_prev, c_prev, xt, parameters)
    
    注意：ft/it/ot 代表忘记/更新/输出门，cct 代表候选值（c 波浪线），
          c 代表单元状态（记忆）
    """

    # 从"parameters"中检索参数
    Wf = parameters["Wf"] # 忘记门权重
    bf = parameters["bf"]
    Wi = parameters["Wi"] # 更新门权重（注意变量名称）
    bi = parameters["bi"] # （注意变量名称）
    Wc = parameters["Wc"] # 候选值权重
    bc = parameters["bc"]
    Wo = parameters["Wo"] # 输出门权重
    bo = parameters["bo"]
    Wy = parameters["Wy"] # 预测权重
    by = parameters["by"]
    
    # 从 xt 和 Wy 的形状中检索尺寸
    n_x, m = xt.shape
    n_y, n_a = Wy.shape

    ### START CODE HERE ###
    # 连接 a_prev 和 xt（约 1 行）
    concat = None

    # 使用图 (4) 给出的公式计算 ft、it、cct、c_next、ot、a_next 的值（约 6 行）
    ft = None
    it = None
    cct = None
    c_next = None
    ot = None
    a_next = None
    
    # 计算 LSTM 单元的预测（约 1 行）
    yt_pred = None
    ### END CODE HERE ###

    # 将反向传播所需的值存储在缓存中
    cache = (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters)

    return a_next, c_next, yt_pred, cache

In [ ]:
np.random.seed(1)
xt_tmp = np.random.randn(3, 10)
a_prev_tmp = np.random.randn(5, 10)
c_prev_tmp = np.random.randn(5, 10)
parameters_tmp = {}
parameters_tmp['Wf'] = np.random.randn(5, 5 + 3)
parameters_tmp['bf'] = np.random.randn(5, 1)
parameters_tmp['Wi'] = np.random.randn(5, 5 + 3)
parameters_tmp['bi'] = np.random.randn(5, 1)
parameters_tmp['Wo'] = np.random.randn(5, 5 + 3)
parameters_tmp['bo'] = np.random.randn(5, 1)
parameters_tmp['Wc'] = np.random.randn(5, 5 + 3)
parameters_tmp['bc'] = np.random.randn(5, 1)
parameters_tmp['Wy'] = np.random.randn(2, 5)
parameters_tmp['by'] = np.random.randn(2, 1)

a_next_tmp, c_next_tmp, yt_tmp, cache_tmp = lstm_cell_forward(xt_tmp, a_prev_tmp, c_prev_tmp, parameters_tmp)

print("a_next[4] = \n", a_next_tmp[4])
print("a_next.shape = ", a_next_tmp.shape)
print("c_next[2] = \n", c_next_tmp[2])
print("c_next.shape = ", c_next_tmp.shape)
print("yt[1] =", yt_tmp[1])
print("yt.shape = ", yt_tmp.shape)
print("cache[1][3] =\n", cache_tmp[1][3])
print("len(cache) = ", len(cache_tmp))

# 单元测试
lstm_cell_forward_test(lstm_cell_forward)

**预期输出**：

```Python
a_next[4] = 
 [-0.66408471  0.0036921   0.02088357  0.22834167 -0.85575339  0.00138482
  0.76566531  0.34631421 -0.00215674  0.43827275]
a_next.shape =  (5, 10)
c_next[2] = 
 [ 0.63267805  1.00570849  0.35504474  0.20690913 -1.64566718  0.11832942
  0.76449811 -0.0981561  -0.74348425 -0.26810932]
c_next.shape =  (5, 10)
yt[1] = [ 0.79913913  0.15986619  0.22412122  0.15606108  0.97057211  0.31146381
  0.00943007  0.12666353  0.39380172  0.07828381]
yt.shape =  (2, 10)
cache[1][3] =
 [-0.16263996  1.03729328  0.72938082 -0.54101719  0.02752074 -0.30821874
  0.07651101 -1.03752894  1.41219977 -0.37647422]
len(cache) =  10
```

<a name='2-2'></a>
### 2.2 - LSTM 的前向传播

现在你已经实现了 LSTM 的一个步骤，你可以使用 for 循环对其进行迭代，以处理一系列 $T_x$ 输入。 

<img src="images/LSTM_rnn.png" style="width:500px;height:300px;">
<div style="text-align:center;color:purple;"><b>图5</b>：多个时间步上的LSTM。</div>

<a name='ex-4'></a>    
### 练习 4 - lstm_forward
    
实现 `lstm_forward()` 以在 $T_x$ 时间步上运行 LSTM。 

**说明**
* 从变量的形状中获取尺寸 $n_x, n_a, n_y, m, T_x$：`x` 和 `parameters`
* 初始化 3D 张量 $a$、$c$ 和 $y$
    - $a$：隐藏状态，形状$(n_{a}, m, T_{x})$
    - $c$：单元状态，形状 $(n_{a}, m, T_{x})$
    - $y$：预测，形状$(n_{y}, m, T_{x})$（注意本例中是$T_{y} = T_{x}$）
    - **注意** 将一个变量设置为等于另一个变量是“按引用复制”。  换句话说，不要执行“c = a”，否则这两个变量都指向同一个基础变量。
* 初始化2D张量$a^{\langle t \rangle}$ 
    - $a^{\langle t \rangle}$ 存储时间步 $t$ 的隐藏状态。  变量名称为 `a_next`。
    - $a^{\langle 0 \rangle}$，即时间步0处的初始隐藏状态，在调用函数时传入。变量名称为 `a0`。
    - $a^{\langle t \rangle}$ 和 $a^{\langle 0 \rangle}$ 表示单个时间步，因此它们都具有 $(n_{a}, m)$ 的形状 
    - 初始化 $a^{\langle t \rangle}$ 通过将其设置为初始值隐藏状态 ($a^{\langle 0 \rangle}$）被传递到函数.
* 用零初始化 $c^{\langle t \rangle}$。 
    - 变量名为 `c_next`
    - $c^{\langle t \rangle}$ 表示单个时间步，因此其形状为 $(n_{a}, m)$
    - **注意**：创建 `c_next` 作为自己的变量，并在内存中拥有自己的位置。  不要将其初始化为 3D 张量 $c$ 的切片。  换句话说，**不要**执行 `c_next = c[:,:,0]`。
* 对于每个时间步，执行以下操作：
    - 从 3D 张量 $x$，在时间步 $t$ 获取 2D 切片 $x^{\langle t \rangle}$
    - 调用之前定义的`lstm_cell_forward` 函数，获取隐藏状态、单元状态、预测和缓存
    - 将隐藏状态、单元状态和预测（2D 张量）存储在 3D 张量内
    - 将缓存追加到缓存列表中

In [ ]:
# UNQ_C4 (UNIQUE CELL IDENTIFIER, DO NOT EDIT)
# GRADED FUNCTION: lstm_forward

def lstm_forward(x, a0, parameters):
    """
    使用图 (4) 中描述的 LSTM 单元实现循环神经网络的前向传播。

    参数：
    x -- 每个时间步的输入数据，形状为 (n_x, m, T_x)。
    a0 -- 初始隐藏状态，形状为 (n_a, m)
    parameters -- Python字典包含：
                        Wf -- 遗忘门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        bf -- 遗忘门的偏置，NumPy 数组，形状为 (n_a, 1)
                        Wi -- 更新门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        bi -- 更新门的偏置，形状为 (n_a, 1) 的 NumPy 数组
                        Wc -- 第一个 "tanh"、NumPy 形状数组的权重矩阵 (n_a, n_a + n_x)
                        bc -- 第一个 "tanh"、NumPy 形状数组的偏置 (n_a, 1)
                        Wo -- 输出门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        bo -- 输出门的偏置，NumPy 数组，形状为 (n_a, 1)
                        Wy -- 将隐藏状态与输出相关的权重矩阵，形状为 (n_y, n_a) 的 NumPy 数组
                        by -- 输出层的偏置，将隐藏状态映射至输出时使用，为形状 (n_y, 1) 的 NumPy 数组
                        
    返回：
    a -- 每个时间步的隐藏状态，形状为 (n_a, m, T_x) 的 NumPy 数组
    y -- 每个时间步的预测，NumPy 形状数组 (n_y, m, T_x)
    c -- 单元状态的值，NumPy 数组，形状为 (n_a, m, T_x)
    caches -- 反向传播所需的值的元组，包含（所有缓存的列表，x）
    """

    # 初始化"caches"，它将跟踪所有缓存的列表
    caches = []
    
    ### START CODE HERE ###
    Wy = parameters['Wy'] # 将参数 ['Wy'] 保存在局部变量中，以防学生使用 Wy 而不是参数 ['Wy']
    # 从 x 的形状和参数中检索尺寸['Wy']（≈2 行）
    n_x, m, T_x = None
    n_y, n_a = None
    
    # 用零初始化 "a"、"c" 和 "y"（约 3 行）
    a = None
    c = None
    y = None
    
    # 初始化a_next和c_next（约2行）
    a_next = None
    c_next = None
    
    # 循环所有时间步
    for t in range(T_x):
        # 在时间步 't' 从 3D 输入 'x' 获取 2D 切片 'xt'
        xt = x[:,:,t]
        # 更新下一个隐藏状态，下一个内存状态，计算预测，获取缓存（约1行）
        a_next, c_next, yt, cache = None
        # 将新的"next" 隐藏状态的值保存在一个（≈1行）中
        a[:,:,t] = None
        # 保存下一个单元状态的值（约1行）
        c[:,:,t]  = None
        # 将预测值保存在 y（≈1 行）中
        y[:,:,t] = None
        # 将缓存追加到缓存中（约1行）
        None
        
    ### END CODE HERE ###
    
    # 将反向传播所需的值存储在缓存中
    caches = (caches, x)

    return a, y, c, caches

In [ ]:
np.random.seed(1)
x_tmp = np.random.randn(3, 10, 7)
a0_tmp = np.random.randn(5, 10)
parameters_tmp = {}
parameters_tmp['Wf'] = np.random.randn(5, 5 + 3)
parameters_tmp['bf'] = np.random.randn(5, 1)
parameters_tmp['Wi'] = np.random.randn(5, 5 + 3)
parameters_tmp['bi']= np.random.randn(5, 1)
parameters_tmp['Wo'] = np.random.randn(5, 5 + 3)
parameters_tmp['bo'] = np.random.randn(5, 1)
parameters_tmp['Wc'] = np.random.randn(5, 5 + 3)
parameters_tmp['bc'] = np.random.randn(5, 1)
parameters_tmp['Wy'] = np.random.randn(2, 5)
parameters_tmp['by'] = np.random.randn(2, 1)

a_tmp, y_tmp, c_tmp, caches_tmp = lstm_forward(x_tmp, a0_tmp, parameters_tmp)
print("a[4][3][6] = ", a_tmp[4][3][6])
print("a.shape = ", a_tmp.shape)
print("y[1][4][3] =", y_tmp[1][4][3])
print("y.shape = ", y_tmp.shape)
print("caches[1][1][1] =\n", caches_tmp[1][1][1])
print("c[1][2][1]", c_tmp[1][2][1])
print("len(caches) = ", len(caches_tmp))

# 单元测试    
lstm_forward_test(lstm_forward)

**预期输出**：

```Python
a[4][3][6] =  0.172117767533
a.shape =  (5, 10, 7)
y[1][4][3] = 0.95087346185
y.shape =  (2, 10, 7)
caches[1][1][1] =
 [ 0.82797464  0.23009474  0.76201118 -0.22232814 -0.20075807  0.18656139
  0.41005165]
c[1][2][1] -0.855544916718
len(caches) =  2
```

### 恭喜！ 

你现在已经实现了基本 RNN 和 LSTM 的前向传播。使用深度学习框架时，实现前向传播足以构建实现出色性能的系统。框架将处理剩下的事情。 

<div style="color:blue;">

<b>你应该记住什么</b>：
 
* LSTM 与 RNN 类似，它们都使用隐藏状态来传递信息，但 LSTM 还使用单元状态（就像长期记忆）来帮助处理梯度消失的问题
* LSTM 单元由单元状态（或长期记忆）、隐藏状态（或短期记忆）以及不断更新其输入相关性的 3 个门组成：
    * <b>forget</b> 门，它决定哪些输入单元应该被记住并传递。它是一个值在 0 到 1 之间的张量。 
        * 如果一个单元的值接近0，LSTM将“忘记”先前单元状态中存储的状态。
        * 如果它的值接近 1，LSTM 基本上会记住相应的值。
    * <b>update</b> 门，也是一个包含 0 到 1 之间值的张量。它决定丢弃哪些信息以及添加哪些新信息。
        * 当更新门中的一个单元接近1时，其候选值被传递到隐藏状态。
        * 当更新门中的单位接近0时，它会被阻止传递到隐藏状态。
    * 还有一个 <b>output</b> 门，它决定发送什么作为时间步的输出


</div> 

让我们回顾一下到目前为止你所完成的一切。你有： 

* 用于构建序列模型的符号
* 熟悉基本 RNN 和 LSTM 的架构，并能够描述它们的组件

Notebook 的其余部分是可选的，不会评分，但一如既往，我们鼓励你推动自己的理解！祝你好运，玩得开心。 

<a name='3'></a>    
## 3 - 循环神经网络中的反向传播（可选/不计分）

在现代深度学习框架中，你只需要实现前向传播，框架就会照顾反向传播，所以大多数深度学习工程师不需要关心反向传播的细节。但是，如果你是微积分专家（或者只是好奇）并且想要了解 RNN 中反向传播的详细信息，你可以完成 Notebook 的这个可选部分。 

在此前的[课程](https://www.coursera.org/learn/neural-networks-deep-learning/lecture/0VSHe/derivatives-with-a-computation-graph)中，你通过反向传播计算全连接神经网络的代价相对于参数的导数，并据此更新参数。循环神经网络也采用同样的思路。相关公式较复杂，课堂上未作推导，下面将简要列出供你参考。

本 Notebook 没有实现损失 J 到隐藏状态 a 之间的反向传播路径，这部分包括前向传播中的全连接层和 softmax。这里假设相关梯度已在其他地方计算，并通过 da 参数传入 `rnn_backward`。同时假设损失已经按批量大小 m 做过归一化，因此此处无需再次除以样本数。

此部分是可选的且不计分，因为它更困难且有关其实现的细节较少。请注意，本节仅实现完整路径的关键元素！ 

前进，勇敢的人： 

<a name='3-1'></a>    
### 3.1 - 基本 RNN 反向传播

首先计算基本 RNN 单元的反向传播。然后，在以下部分中，迭代单元格。

<img src="images/rnn_backward_overview_3a_1.png" style="width:500px;height:300px;"> <br>
<div style="text-align:center;color:purple;">

<b>图 6</b>：RNN 单元的反向传播。就像在全连接神经网络中一样，代价函数 $J$ 的导数通过遵循微积分的链式法则，通过 RNN 的时间步进行反向传播。在单元内部，链式法则还用于计算$(\frac{\partial J}{\partial W_{ax}},\frac{\partial J}{\partial W_{aa}},\frac{\partial J}{\partial b})$，以更新参数$(W_{ax}, W_{aa}, b_a)$。该操作可以利用来自前向路径的缓存结果。

</div>

回想一下讲座中代价相对于变量的偏导数的简写是 `dVariable`。例如，$\frac{\partial J}{\partial W_{ax}}$ 为 $dW_{ax}$。这将在其余部分中使用。


<img src="images/rnn_cell_backward_3a_c.png" style="width:800px;height:500px;"> <br>
<div style="text-align:center;color:purple;">

<b>图7</b>：<code>rnn_cell_backward</code>的这种实现<strong>不</strong>包括输出全连接层和softmax，它们包含在<code>rnn_cell_forward</code>。  

$da_{next}$ 是 $\frac{\partial{J}}{\partial a^{\langle t \rangle}}$，包括前一阶段和当前阶段输出逻辑的损耗。以绿色显示的添加内容将是 <code>rnn_backward</code> 实现的一部分。

</div>

##### 方程
要计算 `rnn_cell_backward`，你可以使用以下方程。手动导出它们是一个很好的练习。这里，$*$ 表示逐元素乘法，而没有符号表示矩阵乘法。

$$
\begin{align}
\displaystyle a^{\langle t \rangle} &= \tanh(W_{ax} x^{\langle t \rangle} + W_{aa} a^{\langle t-1 \rangle} + b_{a})\tag{-} \\[8pt]
\displaystyle \frac{\partial \tanh(x)} {\partial x} &= 1 - \tanh^2(x) \tag{-} \\[8pt]
\displaystyle {dtanh} &= da_{next} * ( 1 - \tanh^2(W_{ax}x^{\langle t \rangle}+W_{aa} a^{\langle t-1 \rangle} + b_{a})) \tag{0} \\[8pt]
\displaystyle  {dW_{ax}} &= dtanh \cdot x^{\langle t \rangle T}\tag{1} \\[8pt]
\displaystyle dW_{aa} &= dtanh \cdot a^{\langle t-1 \rangle T}\tag{2} \\[8pt]
\displaystyle db_a& = \sum_{batch}dtanh\tag{3} \\[8pt]
\displaystyle dx^{\langle t \rangle} &= { W_{ax}}^T \cdot dtanh\tag{4} \\[8pt]
\displaystyle da_{prev} &= { W_{aa}}^T \cdot dtanh\tag{5}
\end{align}
$$



<a name='ex-5'></a>
### 练习 5 - rnn_cell_backward

实现`rnn_cell_backward`。

通过执行上述方程可以直接计算结果。但是，你可以选择通过计算“dz”并使用链式法则来简化它们。  
可以通过注意到 $\tanh(W_{ax}x^{\langle t \rangle}+W_{aa} a^{\langle t-1 \rangle} + b_{a})$ 被计算并在前向传播中保存为 `a_next` 来进一步简化。 

计算 `dba` 时，公式中的 batch 表示沿全部 m 个样本求和（axis=1）。请使用 `keepdims = True` 保留该维度。

复习课程 1 [带有计算图的导数](https://www.coursera.org/learn/neural-networks-deep-learning/lecture/0VSHe/derivatives-with-a-computation-graph) 到 [反向传播直觉](https://www.coursera.org/learn/neural-networks-deep-learning/lecture/6dDj7/backpropagation-intuition-optional) 可能是值得的，它使用链式法则将计算分解为步骤。  
矩阵向量导数被描述为 [这里](http://cs231n.stanford.edu/vecDerivs.pdf)，尽管上面的方程包含了所需的变换。

**笔记**： `rnn_cell_backward` **不**包括损失的计算 $y \langle t \rangle$。这被合并到传入的 `da_next`。这与以下内容略有不匹配 `rnn_cell_forward`，其中包括全连接层和 softmax. 

**代码注释**： 
  
$\displaystyle dx^{\langle t \rangle}$ 用 dxt 表示，   
$\displaystyle d W_{ax}$ 用 dWax 表示，   
$\displaystyle da_{prev}$ 表示为 da_prev，    
$\displaystyle dW_{aa}$ 用 dWaa 表示，   
$\displaystyle db_{a}$ 表示为 dba，   
`dz` 不是上面导出的，但学生可以选择导出以简化重复计算。



In [ ]:
# 不计分函数：rnn_cell_backward

def rnn_cell_backward(da_next, cache):
    """
    为 RNN 单元实现反向传播（单时间步）。

    参数：
    da_next -- 损失相对于下一时间步隐藏状态的梯度
    cache -- 包含有用值的 python 字典（rnn_cell_forward() 的输出）

    返回：
    gradients -- python 字典包含：
                        dx -- 输入数据的梯度，形状为 (n_x, m)
                        da_prev -- 先前隐藏状态的梯度，形状为 (n_a, m)
                        dWax -- 输入到隐藏权重的梯度，形状为 (n_a, n_x)
                        dWaa -- 梯度隐藏到隐藏的权重，形状为 (n_a, n_a)
                        dba -- 偏置向量梯度，形状为 (n_a, 1)
    """
    
    # 从缓存中检索值
    (a_next, a_prev, xt, parameters) = cache
    
    # 从参数中检索值
    Wax = parameters["Wax"]
    Waa = parameters["Waa"]
    Wya = parameters["Wya"]
    ba = parameters["ba"]
    by = parameters["by"]

    ### START CODE HERE ###
    # 使用 a_next 和 da_next 计算 dtanh 项的梯度（约 1 行）
    dtanh = None

    # 计算相对于 Wax 的损失梯度（约 2 行）
    dxt = None
    dWax = None

    # 计算相对于 Waa 的梯度（约 2 行）
    da_prev = None
    dWaa = None

    # 计算关于 b（≈1 行）的梯度
    dba = None

    ### END CODE HERE ###
    
    # 将梯度存储在 python 字典中
    gradients = {"dxt": dxt, "da_prev": da_prev, "dWax": dWax, "dWaa": dWaa, "dba": dba}
    
    return gradients

In [ ]:
np.random.seed(1)
xt_tmp = np.random.randn(3,10)
a_prev_tmp = np.random.randn(5,10)
parameters_tmp = {}
parameters_tmp['Wax'] = np.random.randn(5,3)
parameters_tmp['Waa'] = np.random.randn(5,5)
parameters_tmp['Wya'] = np.random.randn(2,5)
parameters_tmp['ba'] = np.random.randn(5,1)
parameters_tmp['by'] = np.random.randn(2,1)

a_next_tmp, yt_tmp, cache_tmp = rnn_cell_forward(xt_tmp, a_prev_tmp, parameters_tmp)

da_next_tmp = np.random.randn(5,10)
gradients_tmp = rnn_cell_backward(da_next_tmp, cache_tmp)
print("gradients[\"dxt\"][1][2] =", gradients_tmp["dxt"][1][2])
print("gradients[\"dxt\"].shape =", gradients_tmp["dxt"].shape)
print("gradients[\"da_prev\"][2][3] =", gradients_tmp["da_prev"][2][3])
print("gradients[\"da_prev\"].shape =", gradients_tmp["da_prev"].shape)
print("gradients[\"dWax\"][3][1] =", gradients_tmp["dWax"][3][1])
print("gradients[\"dWax\"].shape =", gradients_tmp["dWax"].shape)
print("gradients[\"dWaa\"][1][2] =", gradients_tmp["dWaa"][1][2])
print("gradients[\"dWaa\"].shape =", gradients_tmp["dWaa"].shape)
print("gradients[\"dba\"][4] =", gradients_tmp["dba"][4])
print("gradients[\"dba\"].shape =", gradients_tmp["dba"].shape)

**预期输出**：

<table>
    <tr>
        <td>
            <b>梯度["dxt"][1][2]</b> =
        </td>
        <td>
           -1.3872130506
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dxt"].shape</b> =
        </td>
        <td>
           (3, 10)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["da_prev"][2][3]</b> =
        </td>
        <td>
           -0.152399493774
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["da_prev"].shape</b> =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["dWax"][3][1]</b> =
        </td>
        <td>
           0.410772824935
        </td>
    </tr>
            <tr>
        <td>
            <b>gradients["dWax"].shape</b> =
        </td>
        <td>
           (5, 3)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["dWaa"][1][2]</b> = 
        </td>
        <td>
           1.15034506685
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWaa"].shape</b> =
        </td>
        <td>
           (5, 5)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度[“dba”][4]</b> = 
        </td>
        <td>
           [ 0.20023491]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dba"].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
</table>

<a name='ex-6'></a>
### 练习 6 - rnn_backward

在每个时间步 $t$ 计算相对于 $a^{\langle t \rangle}$ 的代价梯度很有用，因为它有助于梯度反向传播到前一个 RNN 单元。为此，你需要从最后开始迭代所有时间步，并且在每个步骤中，递增总体 $db_a$、$dW_{aa}$、$dW_{ax}$ 并存储 $dx$。

**说明**：

实现 `rnn_backward` 函数。首先将返回变量初始化为零，然后循环遍历所有时间步，同时在每个时间步调用 `rnn_cell_backward`，相应地更新其他变量。

* 本 Notebook 未实现从损失 J 到隐藏状态 a 的反向传播路径。
    * 这将包括全连接层和 softmax，它们是前向路径的一部分。 
    * 假设这是在其他地方计算的，结果传递到'da'中的`rnn_backward`。 
    * 调用 `rnn_cell_backward` 时，必须将其与前面阶段的损失结合起来（参见上图 7）。
* 进一步假设损失已针对批量大小（m）进行了调整。
    * 因此，这里不需要除以示例的数量。

In [ ]:
# 不计分函数：rnn_backward

def rnn_backward(da, caches):
    """
    在整个输入数据序列上实现反向传播的 RNN。

    参数：
    da -- 所有隐藏状态的上游梯度，形状为 (n_a, m, T_x)
    caches -- 包含来自前向传播 (rnn_forward) 的信息的元组
    
    返回：
    gradients -- python 字典包含：
                        dx -- 梯度 w.r.t。输入数据，NumPy 形状为 (n_x, m, T_x) 的数组
                        da0 -- 梯度 w.r.t 初始隐藏状态、形状为 (n_a, m) 的数组
                        dWax -- 梯度 w.r.t 输入的权重矩阵，NumPy 形状为 (n_a, n_x) 的数组
                        dWaa -- 梯度 w.r.t 隐藏状态的权重矩阵，NumPy-形状数组 (n_a, n_a)
                        dba -- 梯度 w.r.t 形状为 (n_a, 1) 的偏置
    """
        
    ### START CODE HERE ###
    
    # 从缓存的第一个缓存（t=1）中检索值（约 2 行）
    (caches, x) = caches
    (a1, a0, x1, parameters) = caches[0]
    
    # 从 da 和 x1 的形状中检索尺寸（≈2 行代码）
    n_a, m, T_x = da.shape
    n_x, m = x1.shape 
    
    # 使用正确的尺寸（约 6 行）初始化梯度
    dx = None
    dWax = None
    dWaa = None
    dba = None
    da0 = None
    da_prevt = None
    
    # 循环所有时间步
    for t in reversed(range(T_x)):
        # 在时间步 t 计算梯度。明智地选择要在反向传播步骤中使用的 "da_next" 和 "cache"。 （约1行）
        gradients = None
        # 从梯度检索导数（≈ 1 行）
        dxt, da_prevt, dWaxt, dWaat, dbat = gradients["dxt"], gradients["da_prev"], gradients["dWax"], gradients["dWaa"], gradients["dba"]
        # 通过在时间步 t（≈4 行）添加导数来增加全局导数 w.r.t 参数
        dx[:, :, t] = None  
        dWax += None  
        dWaa += None  
        dba += None  
        
    # 将 da0 设置为 a 的梯度，该值已通过所有时间步（≈1 行）反向传播 
    da0 = None
    ### END CODE HERE ###

    # 将梯度存储在 python 字典中
    gradients = {"dx": dx, "da0": da0, "dWax": dWax, "dWaa": dWaa,"dba": dba}
    
    return gradients

In [ ]:
np.random.seed(1)
x_tmp = np.random.randn(3,10,4)
a0_tmp = np.random.randn(5,10)
parameters_tmp = {}
parameters_tmp['Wax'] = np.random.randn(5,3)
parameters_tmp['Waa'] = np.random.randn(5,5)
parameters_tmp['Wya'] = np.random.randn(2,5)
parameters_tmp['ba'] = np.random.randn(5,1)
parameters_tmp['by'] = np.random.randn(2,1)

a_tmp, y_tmp, caches_tmp = rnn_forward(x_tmp, a0_tmp, parameters_tmp)
da_tmp = np.random.randn(5, 10, 4)
gradients_tmp = rnn_backward(da_tmp, caches_tmp)

print("gradients[\"dx\"][1][2] =", gradients_tmp["dx"][1][2])
print("gradients[\"dx\"].shape =", gradients_tmp["dx"].shape)
print("gradients[\"da0\"][2][3] =", gradients_tmp["da0"][2][3])
print("gradients[\"da0\"].shape =", gradients_tmp["da0"].shape)
print("gradients[\"dWax\"][3][1] =", gradients_tmp["dWax"][3][1])
print("gradients[\"dWax\"].shape =", gradients_tmp["dWax"].shape)
print("gradients[\"dWaa\"][1][2] =", gradients_tmp["dWaa"][1][2])
print("gradients[\"dWaa\"].shape =", gradients_tmp["dWaa"].shape)
print("gradients[\"dba\"][4] =", gradients_tmp["dba"][4])
print("gradients[\"dba\"].shape =", gradients_tmp["dba"].shape)

**预期输出**：

<table>
    <tr>
        <td>
            <b>梯度["dx"][1][2]</b> =
        </td>
        <td>
           [-2.07101689 -0.59255627  0.02466855  0.01483317]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dx"].shape</b> =
        </td>
        <td>
           (3, 10, 4)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["da0"][2][3]</b> =
        </td>
        <td>
           -0.314942375127
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["da0"].shape</b> =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
         <tr>
        <td>
            <b>梯度["dWax"][3][1]</b> =
        </td>
        <td>
           11.2641044965
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWax"].shape</b> =
        </td>
        <td>
           (5, 3)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["dWaa"][1][2]</b> = 
        </td>
        <td>
           2.30333312658
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWaa"].shape</b> =
        </td>
        <td>
           (5, 5)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度[“dba”][4]</b> = 
        </td>
        <td>
           [-0.74747722]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dba"].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
</table>

<a name='3-2'></a>
### 3.2 - LSTM 反向传播

#### 1. 单步反向传播
LSTM 反向传播比前向传播稍微复杂一些。

<img src="images/LSTM_cell_backward_rev3a_c2.png" style="width:500px;height:400px;"> <br>
<div style="text-align:center;color:purple;"><b>图 8</b>：LSTM 单元的反向传播。注意：输出部分虽然属于 <code>lstm_cell_forward</code>，但不包含在 <code>lstm_cell_backward</code> 中。</div>

LSTM 反向传播的方程式如下。 （如果你喜欢微积分练习，请随意尝试自己从头开始推导这些练习。）

#### 2. 门的导数
注意全连接层和激活函数之间的栅极导数 ($\gamma$..) 的位置（见上图）。这有利于下一步计算参数导数。 
$$
\begin{align}
d\gamma_o^{\langle t \rangle} &= da_{next}*\tanh(c_{next}) * \Gamma_o^{\langle t \rangle}*\left(1-\Gamma_o^{\langle t \rangle}\right)\tag{7} \\[8pt]
dp\widetilde{c}^{\langle t \rangle} &= \left(dc_{next}*\Gamma_u^{\langle t \rangle}+ \Gamma_o^{\langle t \rangle}* (1-\tanh^2(c_{next})) * \Gamma_u^{\langle t \rangle} * da_{next} \right) * \left(1-\left(\widetilde c^{\langle t \rangle}\right)^2\right) \tag{8} \\[8pt]
d\gamma_u^{\langle t \rangle} &= \left(dc_{next}*\widetilde{c}^{\langle t \rangle} + \Gamma_o^{\langle t \rangle}* (1-\tanh^2(c_{next})) * \widetilde{c}^{\langle t \rangle} * da_{next}\right)*\Gamma_u^{\langle t \rangle}*\left(1-\Gamma_u^{\langle t \rangle}\right)\tag{9} \\[8pt]
d\gamma_f^{\langle t \rangle} &= \left(dc_{next}* c_{prev} + \Gamma_o^{\langle t \rangle} * (1-\tanh^2(c_{next})) * c_{prev} * da_{next}\right)*\Gamma_f^{\langle t \rangle}*\left(1-\Gamma_f^{\langle t \rangle}\right)\tag{10}
\end{align}
$$

#### 3. 参数导数 

$$
dW_f = d\gamma_f^{\langle t \rangle} \begin{bmatrix} a_{prev} \\ x_t\end{bmatrix}^T \tag{11}
$$

$$
dW_u = d\gamma_u^{\langle t \rangle} \begin{bmatrix} a_{prev} \\ x_t\end{bmatrix}^T \tag{12}
$$

$$
dW_c = dp\widetilde c^{\langle t \rangle} \begin{bmatrix} a_{prev} \\ x_t\end{bmatrix}^T \tag{13}
$$

$$
dW_o = d\gamma_o^{\langle t \rangle} \begin{bmatrix} a_{prev} \\ x_t\end{bmatrix}^T \tag{14}
$$


要计算 $db_f, db_u, db_c, db_o$，你只需分别对 $d\gamma_f^{\langle t \rangle}, d\gamma_u^{\langle t \rangle}, dp\widetilde c^{\langle t \rangle}, d\gamma_o^{\langle t \rangle}$ 上的所有“m”个示例（轴 = 1）求和。请注意，你应该具有 `keepdims = True` 选项。

$$
\displaystyle db_f = \sum_{batch}d\gamma_f^{\langle t \rangle}\tag{15}
$$

$$
\displaystyle db_u = \sum_{batch}d\gamma_u^{\langle t \rangle}\tag{16}
$$

$$
\displaystyle db_c = \sum_{batch}dp\widetilde c^{\langle t \rangle}\tag{17}
$$

$$
\displaystyle db_o = \sum_{batch}d\gamma_o^{\langle t \rangle}\tag{18}
$$


最后，你将计算相对于先前的隐藏状态、先前的内存状态和输入的导数。

$$
da_{prev} = W_f^T d\gamma_f^{\langle t \rangle} + W_u^T   d\gamma_u^{\langle t \rangle}+ W_c^T dp\widetilde c^{\langle t \rangle} + W_o^T d\gamma_o^{\langle t \rangle} \tag{19}
$$


这里，为了考虑串联，方程 19 的权重是第一个 n_a，（即$W_f = W_f[:,:n_a]$ 等...）

$$
dc_{prev} = dc_{next}*\Gamma_f^{\langle t \rangle} + \Gamma_o^{\langle t \rangle} * (1- \tanh^2(c_{next}))*\Gamma_f^{\langle t \rangle}*da_{next} \tag{20}
$$


$$
dx^{\langle t \rangle} = W_f^T d\gamma_f^{\langle t \rangle} + W_u^T  d\gamma_u^{\langle t \rangle}+ W_c^T dp\widetilde c^{\langle t \rangle} + W_o^T d\gamma_o^{\langle t \rangle}\tag{21}
$$


其中方程 21 的权重是从 n_a 到末尾，（即$W_f = W_f[:,n_a:]$ 等...）

<a name='ex-7'></a>
### 练习 7 - lstm_cell_backward

通过实现下面的方程 $7-21$ 来实现 `lstm_cell_backward`。 
  
    
**注意**： 

代码中：

$d\gamma_o^{\langle t \rangle}$ 表示为 `dot`，    
$dp\widetilde{c}^{\langle t \rangle}$ 表示为 `dcct`，  
$d\gamma_u^{\langle t \rangle}$ 表示为 `dit`，  
$d\gamma_f^{\langle t \rangle}$ 表示为 `dft`

In [ ]:
# 不计分函数：lstm_cell_backward

def lstm_cell_backward(da_next, dc_next, cache):
    """
    为 LSTM 单元实现反向传播（单时间步）。

    参数：
    da_next -- 下一个隐藏状态的梯度，形状为 (n_a, m)
    dc_next -- 下一个单元状态的梯度，形状为 (n_a, m)
    cache -- 缓存存储来自前向传播的信息

    返回：
    gradients -- python 字典包含：
                        dxt -- 时间步 t 处输入数据的梯度，形状为 (n_x, m)
                        da_prev -- 梯度 w.r.t。之前的隐藏状态、NumPy 数组，形状为 (n_a, m)
                        dc_prev -- 梯度 w.r.t。先前的内存状态，形状为 (n_a, m, T_x)
                        dWf -- 梯度 w.r.t。遗忘门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        dWi -- 梯度 w.r.t。更新门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        dWc -- 梯度 w.r.t。记忆门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        dWo -- 梯度 w.r.t。输出门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        dbf -- 梯度 w.r.t。遗忘门的偏置，形状为(n_a, 1)
                        dbi -- 梯度 w.r.t。更新门的偏置，形状为 (n_a, 1)
                        dbc -- 梯度 w.r.t。存储器门的偏置，形状为 (n_a, 1)
                        dbo -- 梯度 w.r.t。输出门的偏置，形状为 (n_a, 1)
    """

    # 从 "cache" 检索信息
    (a_next, c_next, a_prev, c_prev, ft, it, cct, ot, xt, parameters) = cache
    
    ### START CODE HERE ###
    # 从 xt 和 a_next 的形状中检索尺寸（≈2 行代码）
    n_x, m = None
    n_a, m = None
    
    # 计算与门相关的导数。它们的值可以通过仔细查看方程（7）至（10）（≈4 行）找到
    dot = None
    dcct = None
    dit = None
    dft = None
    
    # 计算参数相关导数。使用方程(11)-(18)（≈8行）
    dWf = None
    dWi = None
    dWc = None
    dWo = None
    dbf = None
    dbi = None
    dbc = None
    dbo = None

    # 计算导数 w.r.t 先前的隐藏状态、先前的内存状态和输入。使用方程(19)-(21)。 （约3行）
    da_prev = None
    dc_prev = None
    dxt = None
    ### END CODE HERE ###
    
    
    
    # 将梯度保存到字典中
    gradients = {"dxt": dxt, "da_prev": da_prev, "dc_prev": dc_prev, "dWf": dWf,"dbf": dbf, "dWi": dWi,"dbi": dbi,
                "dWc": dWc,"dbc": dbc, "dWo": dWo,"dbo": dbo}

    return gradients

In [ ]:
np.random.seed(1)
xt_tmp = np.random.randn(3,10)
a_prev_tmp = np.random.randn(5,10)
c_prev_tmp = np.random.randn(5,10)
parameters_tmp = {}
parameters_tmp['Wf'] = np.random.randn(5, 5+3)
parameters_tmp['bf'] = np.random.randn(5,1)
parameters_tmp['Wi'] = np.random.randn(5, 5+3)
parameters_tmp['bi'] = np.random.randn(5,1)
parameters_tmp['Wo'] = np.random.randn(5, 5+3)
parameters_tmp['bo'] = np.random.randn(5,1)
parameters_tmp['Wc'] = np.random.randn(5, 5+3)
parameters_tmp['bc'] = np.random.randn(5,1)
parameters_tmp['Wy'] = np.random.randn(2,5)
parameters_tmp['by'] = np.random.randn(2,1)

a_next_tmp, c_next_tmp, yt_tmp, cache_tmp = lstm_cell_forward(xt_tmp, a_prev_tmp, c_prev_tmp, parameters_tmp)

da_next_tmp = np.random.randn(5,10)
dc_next_tmp = np.random.randn(5,10)
gradients_tmp = lstm_cell_backward(da_next_tmp, dc_next_tmp, cache_tmp)
print("gradients[\"dxt\"][1][2] =", gradients_tmp["dxt"][1][2])
print("gradients[\"dxt\"].shape =", gradients_tmp["dxt"].shape)
print("gradients[\"da_prev\"][2][3] =", gradients_tmp["da_prev"][2][3])
print("gradients[\"da_prev\"].shape =", gradients_tmp["da_prev"].shape)
print("gradients[\"dc_prev\"][2][3] =", gradients_tmp["dc_prev"][2][3])
print("gradients[\"dc_prev\"].shape =", gradients_tmp["dc_prev"].shape)
print("gradients[\"dWf\"][3][1] =", gradients_tmp["dWf"][3][1])
print("gradients[\"dWf\"].shape =", gradients_tmp["dWf"].shape)
print("gradients[\"dWi\"][1][2] =", gradients_tmp["dWi"][1][2])
print("gradients[\"dWi\"].shape =", gradients_tmp["dWi"].shape)
print("gradients[\"dWc\"][3][1] =", gradients_tmp["dWc"][3][1])
print("gradients[\"dWc\"].shape =", gradients_tmp["dWc"].shape)
print("gradients[\"dWo\"][1][2] =", gradients_tmp["dWo"][1][2])
print("gradients[\"dWo\"].shape =", gradients_tmp["dWo"].shape)
print("gradients[\"dbf\"][4] =", gradients_tmp["dbf"][4])
print("gradients[\"dbf\"].shape =", gradients_tmp["dbf"].shape)
print("gradients[\"dbi\"][4] =", gradients_tmp["dbi"][4])
print("gradients[\"dbi\"].shape =", gradients_tmp["dbi"].shape)
print("gradients[\"dbc\"][4] =", gradients_tmp["dbc"][4])
print("gradients[\"dbc\"].shape =", gradients_tmp["dbc"].shape)
print("gradients[\"dbo\"][4] =", gradients_tmp["dbo"][4])
print("gradients[\"dbo\"].shape =", gradients_tmp["dbo"].shape)

**预期输出**：

<table>
    <tr>
        <td>
            <b>梯度["dxt"][1][2]</b> =
        </td>
        <td>
           3.23055911511
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dxt"].shape</b> =
        </td>
        <td>
           (3, 10)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["da_prev"][2][3]</b> =
        </td>
        <td>
           -0.0639621419711
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["da_prev"].shape</b> =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
         <tr>
        <td>
            <b>梯度["dc_prev"][2][3]</b> =
        </td>
        <td>
           0.797522038797
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["dc_prev"].shape</b> =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["dWf"][3][1]</b> = 
        </td>
        <td>
           -0.147954838164
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWf"].shape</b> =
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["dWi"][1][2]</b> = 
        </td>
        <td>
           1.05749805523
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWi"].shape</b> = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
            <b>梯度["dWc"][3][1]</b> = 
        </td>
        <td>
           2.30456216369
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWc"].shape</b> = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
            <b>梯度["dWo"][1][2]</b> = 
        </td>
        <td>
           0.331311595289
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWo"].shape</b> = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
            <b>梯度[“dbf”][4]</b> = 
        </td>
        <td>
           [ 0.18864637]
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度[“dbf”].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
    <tr>
        <td>
            <b>梯度[“dbi”][4]</b> = 
        </td>
        <td>
           [-0.40142491]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dbi"].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度[“dbc”][4]</b> = 
        </td>
        <td>
           [ 0.25587763]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dbc"].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度[“dbo”][4]</b> = 
        </td>
        <td>
           [ 0.13893342]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dbo"].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
</table>

<a name='3-3'></a>
### 3.3 LSTM 循环神经网络的反向传播

这部分与你上面实现的 `rnn_backward` 函数非常相似。你将首先创建与返回变量具有相同维度的变量。然后，你将从末尾开始迭代所有时间步，并在每次迭代时调用你为 LSTM 实现的一步函数。然后，你将通过单独求和来更新参数。最后返回一个带有新梯度的字典。 

<a name='ex-8'></a>
### 练习 8 - lstm_backward

实现 `lstm_backward` 函数。

**说明**：创建一个从 $T_x$ 开始并向后进行的 for 循环。对于每个步骤，调用 `lstm_cell_backward` 并通过添加新的梯度来更新旧的梯度。请注意，`dxt` 未更新，但已存储。

In [ ]:
# 不计分函数：lstm_backward

def lstm_backward(da, caches):
    
    """
    使用 LSTM 单元（在整个序列上）为 RNN 实现反向传播。

    参数：
    da -- 梯度 w.r.t 隐藏状态、形状为 (n_a, m, T_x) 的数组
    caches -- 缓存存储前向传播 (lstm_forward) 的信息

    返回：
    gradients -- python 字典包含：
                        dx -- 输入的梯度，形状为 (n_x, m, T_x)
                        da0 -- 梯度 w.r.t。之前的隐藏状态、NumPy 数组，形状为 (n_a, m)
                        dWf -- 梯度 w.r.t。遗忘门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        dWi -- 梯度 w.r.t。更新门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        dWc -- 梯度 w.r.t。记忆门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        dWo -- 梯度 w.r.t。输出门的权重矩阵，形状为 (n_a, n_a + n_x) 的 NumPy 数组
                        dbf -- 梯度 w.r.t。遗忘门的偏置，形状为(n_a, 1)
                        dbi -- 梯度 w.r.t。更新门的偏置，形状为 (n_a, 1)
                        dbc -- 梯度 w.r.t。存储器门的偏置，形状为 (n_a, 1)
                        dbo -- 梯度 w.r.t。输出门的偏置，形状为 (n_a, 1)
    """

    # 从缓存的第一个缓存（t=1）中检索值。
    (caches, x) = caches
    (a1, c1, a0, c0, f1, i1, cc1, o1, x1, parameters) = caches[0]
    
    ### START CODE HERE ###
    # 从 da 和 x1 的形状中检索尺寸（≈2 行代码）
    n_a, m, T_x = None
    n_x, m = None
    
    # 用正确的尺寸（约12行）初始化梯度
    dx = None
    da0 = None
    da_prevt = None
    dc_prevt = None
    dWf = None
    dWi = None
    dWc = None
    dWo = None
    dbf = None
    dbi = None
    dbc = None
    dbo = None
    
    # 循环整个序列
    for t in reversed(range(None)):
        # 使用 lstm_cell_backward 计算所有梯度。明智地选择 "da_next"（与 Ex 6 相同）。
        gradients = None
        # 将梯度存储或添加到参数上一步的梯度
        da_prevt = None
        dc_prevt = None
        dx[:,:,t] = None
        dWf += None
        dWi += None
        dWc += None
        dWo += None
        dbf += None
        dbi += None
        dbc += None
        dbo += None
    # 将第一个激活值的梯度设置为反向传播的梯度 da_prev。
    da0 = None
    
    ### END CODE HERE ###

    # 将梯度存储在 python 字典中
    gradients = {"dx": dx, "da0": da0, "dWf": dWf,"dbf": dbf, "dWi": dWi,"dbi": dbi,
                "dWc": dWc,"dbc": dbc, "dWo": dWo,"dbo": dbo}
    
    return gradients

In [ ]:
np.random.seed(1)
x_tmp = np.random.randn(3,10,7)
a0_tmp = np.random.randn(5,10)

parameters_tmp = {}
parameters_tmp['Wf'] = np.random.randn(5, 5+3)
parameters_tmp['bf'] = np.random.randn(5,1)
parameters_tmp['Wi'] = np.random.randn(5, 5+3)
parameters_tmp['bi'] = np.random.randn(5,1)
parameters_tmp['Wo'] = np.random.randn(5, 5+3)
parameters_tmp['bo'] = np.random.randn(5,1)
parameters_tmp['Wc'] = np.random.randn(5, 5+3)
parameters_tmp['bc'] = np.random.randn(5,1)
parameters_tmp['Wy'] = np.zeros((2,5))       # 未使用，但 lstm_forward 需要
parameters_tmp['by'] = np.zeros((2,1))       # 未使用，但 lstm_forward 需要

a_tmp, y_tmp, c_tmp, caches_tmp = lstm_forward(x_tmp, a0_tmp, parameters_tmp)

da_tmp = np.random.randn(5, 10, 4)
gradients_tmp = lstm_backward(da_tmp, caches_tmp)

print("gradients[\"dx\"][1][2] =", gradients_tmp["dx"][1][2])
print("gradients[\"dx\"].shape =", gradients_tmp["dx"].shape)
print("gradients[\"da0\"][2][3] =", gradients_tmp["da0"][2][3])
print("gradients[\"da0\"].shape =", gradients_tmp["da0"].shape)
print("gradients[\"dWf\"][3][1] =", gradients_tmp["dWf"][3][1])
print("gradients[\"dWf\"].shape =", gradients_tmp["dWf"].shape)
print("gradients[\"dWi\"][1][2] =", gradients_tmp["dWi"][1][2])
print("gradients[\"dWi\"].shape =", gradients_tmp["dWi"].shape)
print("gradients[\"dWc\"][3][1] =", gradients_tmp["dWc"][3][1])
print("gradients[\"dWc\"].shape =", gradients_tmp["dWc"].shape)
print("gradients[\"dWo\"][1][2] =", gradients_tmp["dWo"][1][2])
print("gradients[\"dWo\"].shape =", gradients_tmp["dWo"].shape)
print("gradients[\"dbf\"][4] =", gradients_tmp["dbf"][4])
print("gradients[\"dbf\"].shape =", gradients_tmp["dbf"].shape)
print("gradients[\"dbi\"][4] =", gradients_tmp["dbi"][4])
print("gradients[\"dbi\"].shape =", gradients_tmp["dbi"].shape)
print("gradients[\"dbc\"][4] =", gradients_tmp["dbc"][4])
print("gradients[\"dbc\"].shape =", gradients_tmp["dbc"].shape)
print("gradients[\"dbo\"][4] =", gradients_tmp["dbo"][4])
print("gradients[\"dbo\"].shape =", gradients_tmp["dbo"].shape)

**预期输出**：

<table>
    <tr>
        <td>
            <b>梯度["dx"][1][2]</b> =
        </td>
        <td>
           [0.00218254  0.28205375 -0.48292508 -0.43281115]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dx"].shape</b> =
        </td>
        <td>
           (3, 10, 4)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["da0"][2][3]</b> =
        </td>
        <td>
           0.312770310257
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["da0"].shape</b> =
        </td>
        <td>
           (5, 10)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["dWf"][3][1]</b> = 
        </td>
        <td>
           -0.0809802310938
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWf"].shape</b> =
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度["dWi"][1][2]</b> = 
        </td>
        <td>
           0.40512433093
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWi"].shape</b> = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
            <b>梯度["dWc"][3][1]</b> = 
        </td>
        <td>
           -0.0793746735512
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWc"].shape</b> = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
            <b>梯度["dWo"][1][2]</b> = 
        </td>
        <td>
           0.038948775763
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dWo"].shape</b> = 
        </td>
        <td>
           (5, 8)
        </td>
    </tr>
    <tr>
        <td>
            <b>梯度[“dbf”][4]</b> = 
        </td>
        <td>
           [-0.15745657]
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度[“dbf”].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
    <tr>
        <td>
            <b>梯度[“dbi”][4]</b> = 
        </td>
        <td>
           [-0.50848333]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dbi"].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度[“dbc”][4]</b> = 
        </td>
        <td>
           [-0.42510818]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dbc"].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
        <tr>
        <td>
            <b>梯度[“dbo”][4]</b> = 
        </td>
        <td>
           [ -0.17958196]
        </td>
    </tr>
        <tr>
        <td>
            <b>gradients["dbo"].shape</b> = 
        </td>
        <td>
           (5, 1)
        </td>
    </tr>
</table>

### 恭喜你完成此作业！ 

现在你了解了循环神经网络的工作原理了！在下一个练习中，你将使用 RNN 构建字符级语言模型。到时见！ 